<a href="https://colab.research.google.com/github/call-me-rodney/sea-level-monitor/blob/main/ee_sealevel_monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEA LEVEL MONITOR
- Institution: Makerere University
- Course: BCSC
- Course Code: CSC2201
- Year: 2
- Instructor: Dr. Lilian

## GROUP MEMBERS.

| Name | Student Number | Registration Number |
| --- | --- | --- |
| Kawuma Rodney | 2400705564 | 24/U/05564/PS |
| Kamwine Jonan | 2400705257 | 24/U/05257/EVE |
| Twebaze Rodlin | 2400726022 | 24/U/26022/EVE |
| Musinguzi Dennis | 2400707425 | 24/U/07425/PS |

## INTRODUCTION
This project aims to utilize machine learning to detect and measure sea level change on a water body of choice. It utilizes imagery obtained from the Sentinel 2 satellite obtained through google earth engine. As you will see below, the model is capable of taking two images of the same waterbody from two different time periods, and identify the extent of sea level change, potraying it to us the user through an adequate amount of graphs and imagery.

## IMPORT ALL IMPORTANT PROJECT DEPENDENCIES

In [ ]:
# Google Earth Engine
import ee
import geemap
# Google Drive
from google.colab import drive
# Image Augmentation
!pip install albumentations --quiet
import albumentations as A
import json, glob
# Dataset Generation
import torch
from torch.utils.data import Dataset, DataLoader
# Model Training
!pip install segmentation_models_pytorch --quiet
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
# Data manipulation
!pip install rasterio --quiet
import rasterio, matplotlib.pyplot as plt, numpy as np, requests, os, pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from itertools import product

## INITIALIZE GCP EARTH ENGINE API

In [ ]:
# Authenticate and initialize earth engine project
ee.Authenticate(auth_mode="colab")
ee.Initialize(project='ee-group-waterproject')

## MOUNT DRIVE FOR DATA STORAGE

In [ ]:
drive.mount('/content/drive')
base = '/content/drive/MyDrive/WaterProject'

## SAMPLE OF THE IMAGE, IMAGE FETCHING AND DRIVE SAVING PROCESS

In [ ]:
# Define the area of interest.
roi = ee.Geometry.Rectangle([32.55, 0.18, 32.77, 0.40])  # Murchison Bay

# Define image parametres
img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
       .filterBounds(roi)
       .filterDate('2023-07-01', '2023-09-30')
       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
       .median()
       .clip(roi)
       .select(['B4', 'B3', 'B2', 'B8'])  # Red, Green, Blue, NIR
       .toUint16())

# Get the download URL
url = img.getDownloadURL({
    'name': 'murchison_test',
    'scale': 30,           # coarser to keep file small for the test
    'region': roi,
    'crs': 'EPSG:4326',
    'format': 'GEO_TIFF'
})
print(f"Download URL: {url}")

# Save to drive
out_path = f'{base}/murchison_test.tif'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'wb') as f:
    f.write(requests.get(url).content)
print('Saved:', os.path.getsize(out_path), 'bytes')

# Finally, read and plot the results.
with rasterio.open(out_path) as src:
    rgb = src.read([1, 2, 3])  # B4, B3, B2 → R, G, B
    print('Shape:', rgb.shape, 'CRS:', src.crs)

# Stretch to display range (Sentinel-2 reflectance ~0–10000)
rgb = np.transpose(rgb, (1, 2, 0)).astype(np.float32)
rgb = np.clip(rgb / 3000, 0, 1)

plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.title('Murchison Bay — smoke test')
plt.axis('off')
plt.show()

## OUTLINE OF BODIES TO BE USED FOR MODEL TRAINING AND EVALUATION.
### CANDIDATES.

| Lake | Bbox (lng_min, lat_min, lng_max, lat_max) | No of Scenes | Use | Why interesting |
| --- | --- | --- | --- | --- |
| Murchison Bay (Kampala) | 32.55, 0.18, 32.77, 0.40 | 3 | Train | Urban, & polluted |
| Lake Kyoga shoreline | 32.70, 1.35, 33.00, 1.65 | 13 | Train | Papyrus wetland, with fuzzy boundary |
| Lake Albert (Hoima) | 30.80, 1.30, 31.10, 1.60 | 6 | Train | Large rift lake, with flat shores |
| Lake Bunyonyi | 29.88, -1.35, 29.98, -1.25 | 5 | Validation | Sharp shores, with clear water |
| Lake Wamala | 31.80, 0.30, 32.05, 0.55 | 5 | Test | Documented shrinking — proof of work |
| Lake Mburo | 30.90,-0.70, 30.97,-0.62 | 6 | Test | Well documented |

## FETCHING AND STORING REQUIRED IMAGE DATA INTO DRIVE

In [ ]:
# Note: scales vary due to Earth Engine's 50 MB download-size constraint
image_fields = [
    {"name": "murchisonBay_2023", "roi": [32.55, 0.18, 32.77, 0.40], "scale": 12},
    {"name": "kyoga_2023", "roi": [32.70, 1.35, 33.00, 1.65], "scale": 16.4},
    {"name": "albert_2023", "roi": [30.80, 1.30, 31.10, 1.60], "scale": 16.5},
    {"name": "bunyonyi_2023", "roi": [29.88, -1.35, 29.98, -1.25], "scale": 10},
    {"name": "wamala_2023", "roi": [31.80, 0.30, 32.05, 0.55], "scale": 13.6},
    {"name":"mburo_2023", "roi":[30.90,-0.70, 30.97,-0.62], "scale": 10}
]

def maskClouds(img):
    qa = img.select('QA60')
    cloud = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus = qa.bitwiseAnd(1 << 11).eq(0)
    return img.updateMask(cloud.And(cirrus))

os.makedirs(f'{base}/raw', exist_ok=True)

for i in image_fields:
    roi = ee.Geometry.Rectangle(i["roi"])
    name = i["name"]
    scale = i["scale"]

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(roi)
          .filterDate('2023-06-01', '2023-09-30')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .map(maskClouds))

    n_scenes = s2.size().getInfo()
    print(f'{name}: {n_scenes} scenes')

    img = s2.median().clip(roi).select(['B4', 'B3', 'B2', 'B8']).toUint16()

    download_url = img.getDownloadURL({
        'name': name,
        'scale': scale,
        'region': roi,
        'crs': 'EPSG:4326',
        'format': 'GEO_TIFF',
        'maxPixels': 1e9,
    })

    # Download with a basic integrity check
    resp = requests.get(download_url)
    out_path = f'{base}/raw/{name}.tif'

    if resp.status_code != 200 or len(resp.content) < 1000:
        print(f'Download failed for {name} '
              f'(status {resp.status_code}, {len(resp.content)} bytes).')
        continue

    with open(out_path, 'wb') as f:
        f.write(resp.content)
    print(f'  Saved: {os.path.getsize(out_path):,} bytes → {out_path}')

## TRAINING DATA INSPECTION


In [ ]:
# Minor check to ensure all retrieved images are up to par with the needs of the task at hand
raw_dir = f'{base}/raw'
files = sorted(f for f in os.listdir(raw_dir) if f.endswith('.tiff'))
print(files)

fig, axes = plt.subplots(1, len(files), figsize=(5 * len(files), 5))
if len(files) == 1: axes = [axes]

for ax, f in zip(axes, files):
    with rasterio.open(os.path.join(raw_dir, f)) as src:
        rgb = src.read([1, 2, 3]).astype(np.float32)
    rgb = np.clip(np.transpose(rgb, (1, 2, 0)) / 3000, 0, 1)
    ax.imshow(rgb)
    ax.set_title(f)
    ax.axis('off')
plt.tight_layout(); plt.show()

## PREPARE AND SPLIT IMAGE DATA


In [ ]:
# This is to produce image chunks that the model can train, validate, and test on
ndwi_dir  = f'{base}/ndwi'
mask_dir  = f'{base}/masks'
tiles_dir = f'{base}/tiles'

for d in [ndwi_dir, mask_dir, tiles_dir]:
    os.makedirs(d, exist_ok=True)

THRESHOLD = 0.0
TILE      = 256
STRIDE    = 128

for f in sorted(os.listdir(raw_dir)):
    if not f.endswith('.tiff'): continue
    name = f.replace('.tiff', '')

    # 1. Read raw bands
    with rasterio.open(os.path.join(raw_dir, f)) as src:
        bands = src.read().astype(np.float32)   # (4, H, W) — B4, B3, B2, B8
        meta  = src.meta.copy()

    # 2. Compute NDWI = (Green - NIR) / (Green + NIR)
    green, nir = bands[1], bands[3]
    ndwi = (green - nir) / (green + nir + 1e-6)

    ndwi_meta = meta.copy(); ndwi_meta.update(count=1, dtype='float32')
    with rasterio.open(os.path.join(ndwi_dir, f'{name}_ndwi.tiff'), 'w', **ndwi_meta) as dst:
        dst.write(ndwi[np.newaxis].astype(np.float32))

    # 3. Threshold to binary mask
    msk = (ndwi > THRESHOLD).astype(np.uint8)

    msk_meta = meta.copy(); msk_meta.update(count=1, dtype='uint8', nodata=255)
    with rasterio.open(os.path.join(mask_dir, f'{name}_mask.tiff'), 'w', **msk_meta) as dst:
        dst.write(msk[np.newaxis])

    # 4. Chip into 256×256 tiles
    img = bands  # already loaded
    H, W = msk.shape
    region_dir = os.path.join(tiles_dir, name)
    os.makedirs(region_dir, exist_ok=True)

    idx = 0
    for y, x in product(range(0, H - TILE + 1, STRIDE),
                        range(0, W - TILE + 1, STRIDE)):
        img_tile = img[:, y:y+TILE, x:x+TILE]
        msk_tile = msk[y:y+TILE, x:x+TILE]
        if img_tile.max() == 0:
            continue
        np.save(os.path.join(region_dir, f'{idx:05d}_img.npy'), img_tile)
        np.save(os.path.join(region_dir, f'{idx:05d}_msk.npy'), msk_tile)
        idx += 1

    print(f'{name}: NDWI ✓ | mask ({msk.mean()*100:.1f}% water) ✓ | {idx} tiles ✓')

In [ ]:
# Create splits manifest
splits = {
    'train': ['murchisonBay_2023', 'kyoga_2023', 'albert_2023'],
    'val':   ['bunyonyi_2023'],
    'test':  ['wamala_2023'],
}

with open(f'{base}/splits.json', 'w') as f:
    json.dump(splits, f, indent=2)

print('Splits written.')

#Verify integrity of splits maniifest
with open(f'{base}/splits.json') as f:
    splits = json.load(f)

tiles_dir = f'{base}/tiles'
for split, regions in splits.items():
    total = 0
    for r in regions:
        n = len([f for f in os.listdir(os.path.join(tiles_dir, r))
                 if f.endswith('_img.npy')])
        total += n
        print(f'  {r}: {n} tiles')
    print(f'{split.upper()} total: {total}\n')

## DATASET GENERATION, SPLITTING AND AUGMENTATION

In [ ]:
splits_path = f'{base}/splits.json'

# 2. Verify upstream outputs
assert os.path.isdir(tiles_dir), 'tiles/ missing — run chipping task first.'
assert os.path.isfile(splits_path), 'splits.json missing — run splits task first.'

with open(splits_path) as f:
    splits = json.load(f)

# Verify no region appears in multiple splits (split-by-water-body integrity)
all_regions = [r for regions in splits.values() for r in regions]
assert len(all_regions) == len(set(all_regions)), \
    'A region appears in multiple splits — check splits.json for leakage.'
print('Splits clean. No region leakage.\n')

In [ ]:
# Train: flips/rotations/brightness — water-segmentation-safe
# Val/Test: no augmentation required
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(std_range=(0.0, 0.03), p=0.2),
])
val_transform = None

# 4. Dataset class
class WaterDataset(Dataset):
    def __init__(self, tiles_dir, regions, transform=None):
        self.transform = transform
        self.pairs = []
        for r in regions:
            region_path = os.path.join(tiles_dir, r)
            for img_p in sorted(glob.glob(os.path.join(region_path, '*_img.npy'))):
                msk_p = img_p.replace('_img.npy', '_msk.npy')
                self.pairs.append((img_p, msk_p))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img_p, msk_p = self.pairs[i]
        img = np.load(img_p).astype(np.float32) / 10000.0   # (4, H, W) → 0–1
        msk = np.load(msk_p).astype(np.float32)             # (H, W)    → 0/1

        if self.transform is not None:
            # albumentations expects (H, W, C); convert, transform, convert back
            img_hwc = np.transpose(img, (1, 2, 0))
            t = self.transform(image=img_hwc, mask=msk)
            img = np.transpose(t['image'], (2, 0, 1))
            msk = t['mask']

        return torch.from_numpy(img), torch.from_numpy(msk).unsqueeze(0)

# 5. Build datasets per split
train_ds = WaterDataset(tiles_dir, splits['train'], transform=train_transform)
val_ds   = WaterDataset(tiles_dir, splits['val'],   transform=val_transform)
test_ds  = WaterDataset(tiles_dir, splits['test'],  transform=val_transform)

print(f'Train: {len(train_ds)} tiles')
print(f'Val:   {len(val_ds)} tiles')
print(f'Test:  {len(test_ds)} tiles\n')

In [ ]:
# 6. DataLoaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                          num_workers=2, pin_memory=True)

# 7. Smoke test: shapes and dtypes
img_batch, msk_batch = next(iter(train_loader))
print(f'Image batch: {tuple(img_batch.shape)}  dtype={img_batch.dtype}')
print(f'Mask  batch: {tuple(msk_batch.shape)}  dtype={msk_batch.dtype}')
print(f'Image range: [{img_batch.min():.3f}, {img_batch.max():.3f}]')
print(f'Mask  unique values: {torch.unique(msk_batch).tolist()}\n')

# 8. Eyeball 8 augmented tiles
N = 8
fig, axes = plt.subplots(N, 3, figsize=(12, 3*N))
for i in range(N):
    img = np.transpose(img_batch[i].numpy(), (1, 2, 0))
    rgb = np.clip(img[:, :, :3] * 3.0, 0, 1)   # display stretch
    msk = msk_batch[i, 0].numpy()

    axes[i, 0].imshow(rgb);                    axes[i, 0].set_title('Image (augmented)')
    axes[i, 1].imshow(msk, cmap='Blues');      axes[i, 1].set_title('Mask (augmented)')
    axes[i, 2].imshow(rgb)
    axes[i, 2].imshow(msk, cmap='Blues', alpha=0.4)
    axes[i, 2].set_title(f'Overlay  ({msk.mean()*100:.1f}% water)')
    for a in axes[i]: a.axis('off')

plt.tight_layout()
plt.show()
print('Dataset, splits, augmentations, DataLoaders all working.')

MODEL INITIALIZATION, TRAINING AND EVALUATION

In [ ]:
# 1. Verify upstream state
assert 'train_loader' in dir() and 'val_loader' in dir(), \
    'train_loader / val_loader missing — run Day 4 cell first.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('No GPU detected. Runtime → Change runtime type → T4 GPU.')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2. Build the U-Net
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=4,            # B4, B3, B2, B8
    classes=1,                # binary water/not-water
).to(device)

print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

# Forward-pass smoke test
with torch.no_grad():
    x = torch.randn(2, 4, 256, 256).to(device)
    out = model(x)
assert out.shape == (2, 1, 256, 256), f'Unexpected output shape: {out.shape}'
print(f'Forward pass ✓  output shape {tuple(out.shape)}\n')

# 3. Loss and metrics
bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

def combo_loss(pred, target):
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)

def compute_metrics(pred_logits, target, threshold=0.5):
    """Pred logits → sigmoid → threshold → IoU/precision/recall."""
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    target = target.float()
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    eps = 1e-6
    return {
        'iou':       (tp / (tp + fp + fn + eps)).item(),
        'precision': (tp / (tp + fp + eps)).item(),
        'recall':    (tp / (tp + fn + eps)).item(),
    }

# 4. Training config
EPOCHS   = 20
LR       = 1e-4
CKPT_DIR = f'{base}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_iou': [],
           'val_precision': [], 'val_recall': []}
best_iou = 0.0

# 5. Training loop
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_losses = []
    for img, msk in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):
        img, msk = img.to(device), msk.to(device)
        pred = model(img)
        loss = combo_loss(pred, msk)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # Validate
    model.eval()
    val_losses, val_ious, val_precs, val_recs = [], [], [], []
    with torch.no_grad():
        for img, msk in val_loader:
            img, msk = img.to(device), msk.to(device)
            pred = model(img)
            val_losses.append(combo_loss(pred, msk).item())
            m = compute_metrics(pred, msk)
            val_ious.append(m['iou'])
            val_precs.append(m['precision'])
            val_recs.append(m['recall'])

    scheduler.step()

    avg_train = sum(train_losses) / len(train_losses)
    avg_vloss = sum(val_losses) / len(val_losses)
    avg_iou   = sum(val_ious) / len(val_ious)
    avg_prec  = sum(val_precs) / len(val_precs)
    avg_rec   = sum(val_recs) / len(val_recs)

    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_vloss)
    history['val_iou'].append(avg_iou)
    history['val_precision'].append(avg_prec)
    history['val_recall'].append(avg_rec)

    print(f'Epoch {epoch+1:2d}: train={avg_train:.4f}  val={avg_vloss:.4f}  '
          f'IoU={avg_iou:.4f}  P={avg_prec:.4f}  R={avg_rec:.4f}')

    # Checkpoint
    ckpt = {
        'epoch': epoch + 1,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'val_iou': avg_iou,
        'history': history,
    }
    torch.save(ckpt, os.path.join(CKPT_DIR, f'epoch_{epoch+1:02d}.pth'))
    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(ckpt, os.path.join(CKPT_DIR, 'best.pth'))
        print(f'   ↳ new best (IoU {best_iou:.4f}) saved')

print(f'\nTraining complete. Best val IoU: {best_iou:.4f}')

# 6. Plot loss + IoU curves
epochs_x = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, history['train_loss'], label='Train', linewidth=2)
axes[0].plot(epochs_x, history['val_loss'],   label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss curves')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['val_iou'],       label='IoU',       linewidth=2, color='green')
axes[1].plot(epochs_x, history['val_precision'], label='Precision', linewidth=1.5, alpha=0.7)
axes[1].plot(epochs_x, history['val_recall'],    label='Recall',    linewidth=1.5, alpha=0.7)
axes[1].axhline(0.85, ls='--', color='red', alpha=0.5, label='Target IoU 0.85')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Metric')
axes[1].set_title('Validation metrics')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{base}/training_curves.png',
            dpi=120, bbox_inches='tight')
plt.show()

print('Day 5 ✓  — model trained, checkpoints saved, curves plotted.')

## Compute IoU, precision, recall on held-out water body

In [ ]:
# Load the best checkpoint
ckpt = torch.load(f'{base}/checkpoints/best.pth')
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]} with val IoU {ckpt["val_iou"]:.4f}')

In [ ]:
# Compute test-set metrics
def evaluate(loader):
    """Aggregate TP/FP/FN over the full set, then compute final metrics."""
    tp = fp = fn = 0
    with torch.no_grad():
        for img, msk in loader:
            img, msk = img.to(device), msk.to(device)
            pred = (torch.sigmoid(model(img)) > 0.5).float()

            tp += (pred * msk).sum().item()
            fp += (pred * (1 - msk)).sum().item()
            fn += ((1 - pred) * msk).sum().item()

    eps = 1e-6
    iou       = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)
    return {'IoU': iou, 'Precision': precision, 'Recall': recall, 'F1': f1}

results = evaluate(test_loader)
print('--- TEST SET (Lake Wamala) ---')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

# Save test results
with open(f'{base}/test_metrics.txt', 'w') as f:
    f.write('Test set: Lake Wamala\n')
    for k, v in results.items():
        f.write(f'{k}: {v:.4f}\n')

## Visualize 10–15 predictions side-by-side with ground truth

In [ ]:
model.eval()
N = 12

# Pull a batch from test loader
img_batch, msk_batch = next(iter(test_loader))
img_batch = img_batch.to(device)

with torch.no_grad():
    pred_batch = (torch.sigmoid(model(img_batch)) > 0.5).float().cpu()

img_batch = img_batch.cpu()

fig, axes = plt.subplots(N, 3, figsize=(12, 3.5*N))
for i in range(N):
    img = np.transpose(img_batch[i].numpy(), (1, 2, 0))
    rgb = np.clip(img[:, :, :3] * 3.0, 0, 1)
    gt   = msk_batch[i, 0].numpy()
    pred = pred_batch[i, 0].numpy()

    # IoU for this single tile
    inter = (pred * gt).sum()
    union = ((pred + gt) > 0).sum()
    tile_iou = inter / (union + 1e-6)

    axes[i, 0].imshow(rgb);                axes[i, 0].set_title('Input')
    axes[i, 1].imshow(gt,   cmap='Blues'); axes[i, 1].set_title('Ground truth')
    axes[i, 2].imshow(pred, cmap='Blues'); axes[i, 2].set_title(f'Prediction (IoU {tile_iou:.2f})')
    for a in axes[i]: a.axis('off')

plt.tight_layout()
plt.savefig(f'{base}/predictions_grid.png', dpi=150, bbox_inches='tight')
plt.show()

# To find specific tiles by IoU
ious = []
for i in range(len(test_ds)):
    img, msk = test_ds[i]
    with torch.no_grad():
        pred = (torch.sigmoid(model(img.unsqueeze(0).to(device))) > 0.5).float().cpu()
    inter = (pred[0,0] * msk[0]).sum().item()
    union = ((pred[0,0] + msk[0]) > 0).sum().item()
    ious.append((i, inter / (union + 1e-6)))

# Best, worst, median
ious.sort(key=lambda x: x[1])
to_show = [ious[0][0], ious[len(ious)//2][0], ious[-1][0]]
print('Will visualize tile indices:', to_show)

## Document failure modes (turbid water? small ponds? cloud shadows?)

In [ ]:
failures = []
for i in range(len(test_ds)):
    img, msk = test_ds[i]
    with torch.no_grad():
        pred = (torch.sigmoid(model(img.unsqueeze(0).to(device))) > 0.5).float().cpu()[0, 0]

    inter = (pred * msk[0]).sum().item()
    union = ((pred + msk[0]) > 0).sum().item()
    iou = inter / (union + 1e-6)

    fp = (pred * (1 - msk[0])).sum().item()
    fn = ((1 - pred) * msk[0]).sum().item()

    if iou < 0.7 or fp > 100 or fn > 100:
        failures.append({'idx': i, 'iou': iou, 'fp': fp, 'fn': fn})

print(f'{len(failures)} potentially problematic tiles found')
failures.sort(key=lambda x: x['iou'])

# Replot
strict_failures = [f for f in failures if f['iou'] < 0.85]
print(f'Tiles with genuinely bad IoU (< 0.85): {len(strict_failures)} / {len(test_ds)}')

really_bad = [f for f in failures if f['iou'] < 0.70]
print(f'Tiles with poor IoU (< 0.70): {len(really_bad)} / {len(test_ds)}')

# Save the failure analysis
import json
with open(f'{base}/failure_analysis.json', 'w') as f:
    json.dump({'failures': failures[:20]}, f, indent=2, default=str)

## Day 8: Iterate model — augmentation, encoder, or training length
Tune certain, parametres to increase IoU, then document what helped and what didn't. Since IoU is high enough, this will be performed after a full notebook evaluation.

## Test on Lake Mburo (genuinely unseen geography)

In [ ]:
# # raw_dir
# # ndwi_dir
# # mask_dir
# # tiles_dir

# THRESHOLD = 0.0
# TILE      = 256
# STRIDE    = 128

# name = "mburo_2023"

# # 1. Read raw bands
# with rasterio.open(os.path.join(raw_dir, "mburo_2023.tiff")) as src:
#     bands = src.read().astype(np.float32)   # (4, H, W) — B4, B3, B2, B8
#     meta  = src.meta.copy()

# # 2. Compute NDWI = (Green - NIR) / (Green + NIR)
# green, nir = bands[1], bands[3]
# ndwi = (green - nir) / (green + nir + 1e-6)

# ndwi_meta = meta.copy(); ndwi_meta.update(count=1, dtype='float32')
# with rasterio.open(os.path.join(ndwi_dir, f'{name}_ndwi.tiff'), 'w', **ndwi_meta) as dst:
#     dst.write(ndwi[np.newaxis].astype(np.float32))

# # 3. Threshold to binary mask
# msk = (ndwi > THRESHOLD).astype(np.uint8)

# msk_meta = meta.copy(); msk_meta.update(count=1, dtype='uint8', nodata=255)
# with rasterio.open(os.path.join(mask_dir, f'{name}_mask.tiff'), 'w', **msk_meta) as dst:
#     dst.write(msk[np.newaxis])

# # 4. Chip into 256×256 tiles
# img = bands  # already loaded
# H, W = msk.shape
# region_dir = os.path.join(tiles_dir, name)
# os.makedirs(region_dir, exist_ok=True)

# idx = 0
# for y, x in product(range(0, H - TILE + 1, STRIDE),
#                     range(0, W - TILE + 1, STRIDE)):
#     img_tile = img[:, y:y+TILE, x:x+TILE]
#     msk_tile = msk[y:y+TILE, x:x+TILE]
#     if img_tile.max() == 0:
#         continue
#     np.save(os.path.join(region_dir, f'{idx:05d}_img.npy'), img_tile)
#     np.save(os.path.join(region_dir, f'{idx:05d}_msk.npy'), msk_tile)
#     idx += 1

# print(f'{name}: NDWI ✓ | mask ({msk.mean()*100:.1f}% water) ✓ | {idx} tiles ✓')

In [ ]:
# Run inference on Mburo
mburo_ds = WaterDataset(tiles_dir, ['mburo_2023'])
mburo_loader = DataLoader(mburo_ds, batch_size=16, shuffle=False, num_workers=2)

mburo_results = evaluate(mburo_loader)
print('=== LAKE MBURO (truly unseen) ===')
for k, v in mburo_results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Visualize 12 tiles for failure analysis
# model.eval()
N = 12

# Pull a batch from test loader
img_batch, msk_batch = next(iter(mburo_loader))
img_batch = img_batch.to(device)

with torch.no_grad():
    pred_batch = (torch.sigmoid(model(img_batch)) > 0.5).float().cpu()

img_batch = img_batch.cpu()

fig, axes = plt.subplots(N, 3, figsize=(12, 3.5*N))
for i in range(N):
    img = np.transpose(img_batch[i].numpy(), (1, 2, 0))
    rgb = np.clip(img[:, :, :3] * 3.0, 0, 1)
    gt   = msk_batch[i, 0].numpy()
    pred = pred_batch[i, 0].numpy()

    # IoU for this single tile
    inter = (pred * gt).sum()
    union = ((pred + gt) > 0).sum()
    tile_iou = inter / (union + 1e-6)

    axes[i, 0].imshow(rgb);                axes[i, 0].set_title('Input')
    axes[i, 1].imshow(gt,   cmap='Blues'); axes[i, 1].set_title('Ground truth')
    axes[i, 2].imshow(pred, cmap='Blues'); axes[i, 2].set_title(f'Prediction (IoU {tile_iou:.2f})')
    for a in axes[i]: a.axis('off')

plt.tight_layout()
plt.savefig(f'{base}/predictions_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare to NDWI baseline
with rasterio.open(f'{base}/raw/mburo_2023.tiff') as src:
    bands = src.read().astype(np.float32)
ndwi = (bands[1] - bands[3]) / (bands[1] + bands[3] + 1e-6)
ndwi_mask = (ndwi > 0).astype(np.uint8)

## ACTUAL SEA LEVEL CHANGE COMPARISON BY MODEL BETWEEN 2019 AND 2023
### SANITY CHECK TO ENSURE PROPER IMAGES WERE PULLED

In [ ]:
raw_2023_dir = raw_dir
raw_2019_dir = f'{base}/raw_2019'
os.makedirs(raw_2019_dir, exist_ok=True)

# Only the lakes used for change detection
CHANGE_LAKES = ['murchisonBay', 'wamala']

geom_lookup = {
    f["name"].replace('_2023', ''): {"roi": f["roi"], "scale": f["scale"]}
    for f in image_fields
}

def maskClouds(img):
    qa = img.select('QA60')
    cloud = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus = qa.bitwiseAnd(1 << 11).eq(0)
    return img.updateMask(cloud.And(cirrus))

for lake in CHANGE_LAKES:
    name = f'{lake}_2019'
    out_path = f'{raw_2019_dir}/{name}.tiff'

    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        print(f'{name}: already downloaded ({os.path.getsize(out_path):,} bytes), skipping')
        continue

    cfg = geom_lookup[lake]
    roi = ee.Geometry.Rectangle(cfg["roi"])
    scale = cfg["scale"]

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(roi)
          .filterDate('2019-06-01', '2019-09-30')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
          .map(maskClouds))

    n_scenes = s2.size().getInfo()
    print(f'{name}: {n_scenes} scenes (scale {scale} m)')
    if n_scenes == 0:
        print(f'No scenes — widen date range or raise cloud threshold, then re-run.')
        continue

    img = s2.median().clip(roi).select(['B4', 'B3', 'B2', 'B8']).toUint16()

    download_url = img.getDownloadURL({
        'name': name,
        'scale': scale,
        'region': roi,
        'crs': 'EPSG:4326',
        'format': 'GEO_TIFF',
        'maxPixels': 1e9,
    })

    resp = requests.get(download_url)
    if resp.status_code != 200 or len(resp.content) < 1000:
        print(f'Download failed (status {resp.status_code}, '
              f'{len(resp.content)} bytes).')
        continue

    with open(out_path, 'wb') as f:
        f.write(resp.content)
    print(f'  Saved: {os.path.getsize(out_path):,} bytes → {out_path}')

def list_scenes(d):
    return sorted(f for f in os.listdir(d) if f.lower().endswith(('.tif', '.tiff')))

files_2023 = list_scenes(raw_2023_dir)
files_2019 = list_scenes(raw_2019_dir)
print('\n2023:', files_2023)
print('2019:', files_2019)

def lake_key(filename):
    return filename.replace('_2023', '').replace('_2019', '').rsplit('.', 1)[0]

paired = {}
for f in files_2023:
    paired.setdefault(lake_key(f), {})['2023'] = f
for f in files_2019:
    paired.setdefault(lake_key(f), {})['2019'] = f

# Keep only lakes that have BOTH years available
paired = {k: v for k, v in paired.items() if '2019' in v and '2023' in v}
print(f'\nLakes with both years: {list(paired.keys())}')

n = len(paired)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 10), squeeze=False)

for col, (lake, years) in enumerate(paired.items()):
    for row, year in enumerate(['2019', '2023']):
        path = os.path.join(raw_2019_dir if year == '2019' else raw_2023_dir,
                            years[year])
        with rasterio.open(path) as src:
            rgb = src.read([1, 2, 3]).astype(np.float32)
            shape = src.shape
        rgb = np.clip(np.transpose(rgb, (1, 2, 0)) / 3000, 0, 1)

        axes[row, col].imshow(rgb)
        axes[row, col].set_title(f'{lake} — {year}  {shape}')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(f'{base}/year_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Run trained model inference on 2019 imagery

In [ ]:
# Sliding-window inference function
def predict_full_scene(model, raw_path, tile_size=256, overlap=64):
    """Run U-Net over a full GeoTIFF using sliding window."""
    with rasterio.open(raw_path) as src:
        img = src.read().astype(np.float32) / 10000.0  # (4, H, W)
        meta = src.meta.copy()

    C, H, W = img.shape
    pad_h = (tile_size - H % tile_size) % tile_size
    pad_w = (tile_size - W % tile_size) % tile_size
    img_p = np.pad(img, ((0,0), (0,pad_h), (0,pad_w)), mode='reflect')

    pred = np.zeros((img_p.shape[1], img_p.shape[2]), dtype=np.float32)
    count = np.zeros_like(pred)

    stride = tile_size - overlap
    model.eval()
    with torch.no_grad():
        for y in range(0, img_p.shape[1] - tile_size + 1, stride):
            for x in range(0, img_p.shape[2] - tile_size + 1, stride):
                tile = img_p[:, y:y+tile_size, x:x+tile_size]
                t = torch.from_numpy(tile).unsqueeze(0).to(device)
                p = torch.sigmoid(model(t))[0, 0].cpu().numpy()
                pred[y:y+tile_size, x:x+tile_size] += p
                count[y:y+tile_size, x:x+tile_size] += 1

    pred = pred / np.maximum(count, 1)
    pred = pred[:H, :W]  # crop padding

    binary = (pred > 0.5).astype(np.uint8)
    return binary, pred, meta

# Run for each scene
for year in ['2019', '2023']:
    for lake in ['murchisonBay', 'wamala']:
        raw_path = f'{base}/raw_{year}/{lake}_{year}.tiff' \
                   if year == '2019' else \
                   f'{base}/raw/{lake}_2023.tiff'

        binary, prob, meta = predict_full_scene(model, raw_path)

        # Save predicted mask as .tiff
        meta.update(count=1, dtype='uint8', nodata=255)
        out_path = f'{base}/predictions/{lake}_{year}_pred.tiff'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(binary[np.newaxis, :, :])

        print(f'{lake} {year}: {binary.mean()*100:.1f}% predicted as water')

In [ ]:
# Sanity Check
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for i, year in enumerate(['2019', '2023']):
    for j, lake in enumerate(['murchisonBay', 'wamala']):
        with rasterio.open(f'{base}/predictions/{lake}_{year}_pred.tiff') as src:
            mask = src.read(1)
        axes[i, j].imshow(mask, cmap='Blues')
        axes[i, j].set_title(f'{lake} {year}')
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()

## Pixel-wise diff between 2019 and 2023 masks

In [ ]:
def compute_change(mask_2019_path, mask_2023_path, out_path):
    with rasterio.open(mask_2019_path) as src:
        m19 = src.read(1)
        meta = src.meta.copy()
    with rasterio.open(mask_2023_path) as src:
        m23 = src.read(1)

    assert m19.shape == m23.shape, f'Shape mismatch: {m19.shape} vs {m23.shape}'

    # Encoding:
    #   0 = persistent land  (0 → 0)
    #   1 = water lost       (1 → 0)
    #   2 = water gained     (0 → 1)
    #   3 = persistent water (1 → 1)
    change = np.zeros_like(m19, dtype=np.uint8)
    change[(m19 == 1) & (m23 == 0)] = 1   # lost
    change[(m19 == 0) & (m23 == 1)] = 2   # gained
    change[(m19 == 1) & (m23 == 1)] = 3   # persistent water
    # persistent land stays 0

    meta.update(count=1, dtype='uint8', nodata=255)
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(change[np.newaxis, :, :])

    return change

pred_dir = f'{base}/predictions'
change_dir = f'{base}/change'
os.makedirs(change_dir, exist_ok=True)

for lake in ['murchisonBay', 'wamala']:
    out = os.path.join(change_dir, f'{lake}_change.tiff')
    change = compute_change(
        os.path.join(pred_dir, f'{lake}_2019_pred.tiff'),
        os.path.join(pred_dir, f'{lake}_2023_pred.tiff'),
        out,
    )

    # Summary
    total = change.size
    print(f'\n=== {lake.upper()} ===')
    print(f'  Persistent land:  {(change == 0).sum() / total * 100:.2f}%')
    print(f'  Water lost:       {(change == 1).sum() / total * 100:.2f}%')
    print(f'  Water gained:     {(change == 2).sum() / total * 100:.2f}%')
    print(f'  Persistent water: {(change == 3).sum() / total * 100:.2f}%')

    # Visualize the change map
    import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

cmap = ListedColormap(['#f0f0f0', '#d62728', '#2ca02c', '#1f77b4'])
labels = ['Persistent land', 'Water lost', 'Water gained', 'Persistent water']

with rasterio.open(f'{base}/change/wamala_change.tiff') as src:
    change = src.read(1)

plt.figure(figsize=(12, 12))
im = plt.imshow(change, cmap=cmap, vmin=0, vmax=3)
cbar = plt.colorbar(im, ticks=[0, 1, 2, 3])
cbar.set_ticklabels(labels)
plt.title('Lake Wamala — change between 2019 and 2023')
plt.axis('off')
plt.savefig(f'{base}/change/wamala_change.png', dpi=150, bbox_inches='tight')
plt.show()

## Translate Pixelwise diff in predictions into actual useful figures

In [ ]:
M_PER_DEG_LAT = 111_320.0  # mean metres per degree of latitude

def per_row_pixel_area_km2(src):
    """
    Per-row pixel area (km²), robust to CRS:
    - Geographic (EPSG:4326): pixel is in degrees; longitude shrinks with
      latitude, so area gets a cos(lat) correction computed per row.
    - Projected (metres): transform is already metric; constant area.
    """
    t = src.transform
    H = src.height
    if src.crs and src.crs.is_geographic:
        dlon = abs(t.a)                              # degrees/pixel in x
        dlat = abs(t.e)                              # degrees/pixel in y
        rows = np.arange(H)
        lats = t.f + (rows + 0.5) * t.e              # latitude at each row centre
        m_lat = dlat * M_PER_DEG_LAT
        m_lon = dlon * M_PER_DEG_LAT * np.cos(np.radians(lats))
        area_m2 = m_lat * m_lon                      # (H,)
    else:
        area_m2 = np.full(H, abs(t.a * t.e))         # already metres
    return area_m2 / 1e6                             # → km²

def class_area_km2(change, cls, row_area_km2):
    """Total km² for one change class, weighting each row by its true area."""
    per_row_counts = (change == cls).sum(axis=1)     # (H,)
    return float((per_row_counts * row_area_km2).sum())

# Optional cross-check against the scales you requested at download time
scale_lookup = {f["name"].replace('_2023', ''): f["scale"] for f in image_fields}

results = []
for lake in ['murchisonBay', 'wamala']:
    with rasterio.open(f'{base}/change/{lake}_change.tiff') as src:
        change   = src.read(1)
        row_area = per_row_pixel_area_km2(src)
        eff_scale = (row_area.mean() * 1e6) ** 0.5   # effective metres/pixel

    lost             = class_area_km2(change, 1, row_area)
    gained           = class_area_km2(change, 2, row_area)
    persistent_water = class_area_km2(change, 3, row_area)

    water_2019 = lost + persistent_water
    water_2023 = gained + persistent_water
    net_change = water_2023 - water_2019
    pct_change = (net_change / water_2019 * 100) if water_2019 > 0 else 0

    print(f'{lake}: effective pixel ≈ {eff_scale:.1f} m '
          f'(requested {scale_lookup.get(lake, "?")} m)')

    results.append({
        'lake': lake,
        'eff_scale_m': round(eff_scale, 1),
        'water_2019_km2': round(water_2019, 3),
        'water_2023_km2': round(water_2023, 3),
        'lost_km2': round(lost, 3),
        'gained_km2': round(gained, 3),
        'net_change_km2': round(net_change, 3),
        'pct_change': round(pct_change, 2),
    })

df = pd.DataFrame(results)
df.to_csv(f'{base}/change_summary.csv', index=False)
print()
print(df.to_string(index=False))

## SANITY CHECK MODEL PREDICTIONS
- Look through official publications from e.g. NEMA to verify validity of my model's prediction.
- Do this after overall notebook completion.

## Bar chart: km² change per water body

In [ ]:
df = pd.read_csv(f'{base}/change_summary.csv')

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(df))
width = 0.35

bars1 = ax.bar(x - width/2, df['water_2019_km2'], width, label='2019', color='#7fb2c7')
bars2 = ax.bar(x + width/2, df['water_2023_km2'], width, label='2023', color='#1f77b4')

ax.set_xticks(x)
ax.set_xticklabels([l.title() for l in df['lake']], fontsize=12)
ax.set_ylabel('Surface water area (km²)', fontsize=12)
ax.set_title('Water surface area change, 2019 vs 2023', fontsize=14, pad=15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Annotate net change above each pair
for i, row in df.iterrows():
    sign = '+' if row['net_change_km2'] >= 0 else ''
    color = 'green' if row['net_change_km2'] >= 0 else 'red'
    ymax = max(row['water_2019_km2'], row['water_2023_km2'])
    ax.annotate(
        f'{sign}{row["net_change_km2"]:.1f} km²\n({sign}{row["pct_change"]:.1f}%)',
        xy=(i, ymax + 1),
        ha='center', fontsize=11, fontweight='bold', color=color,
    )

plt.tight_layout()
plt.savefig(f'{base}/area_change_chart.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Side-by-side before/after maps

In [ ]:
pred_dir   = f'{base}/predictions'
change_dir = f'{base}/change'

cmap_change = ListedColormap(['#f5f5f5', '#d62728', '#2ca02c', '#1f77b4'])
legend_handles = [
    Patch(color='#f5f5f5', label='Persistent land'),
    Patch(color='#d62728', label='Water lost (2019→2023)'),
    Patch(color='#2ca02c', label='Water gained'),
    Patch(color='#1f77b4', label='Persistent water'),
]

for lake in ['murchisonBay', 'wamala']:
    with rasterio.open(f'{pred_dir}/{lake}_2019_pred.tiff') as s: m17 = s.read(1)
    with rasterio.open(f'{pred_dir}/{lake}_2023_pred.tiff') as s: m23 = s.read(1)
    with rasterio.open(f'{change_dir}/{lake}_change.tiff') as s: chg = s.read(1)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    axes[0].imshow(m17, cmap='Blues')
    axes[0].set_title(f'{lake.title()} — 2019', fontsize=14)

    axes[1].imshow(m23, cmap='Blues')
    axes[1].set_title(f'{lake.title()} — 2023', fontsize=14)

    axes[2].imshow(chg, cmap=cmap_change, vmin=0, vmax=3)
    axes[2].set_title(f'{lake.title()} — change', fontsize=14)
    axes[2].legend(handles=legend_handles, loc='lower right', fontsize=9)

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'{change_dir}/{lake}_panel.png', dpi=150, bbox_inches='tight')
    plt.show()